# 05 Robustness

Robustness checks of Chapter 5.

1. FF3 and market model instead of FF5.
2. Pre-2023 / post-2022 split, with and without Third Bridge calls.
3. Estimation windows cleaned of other same-firm events.
4. Placebo: 500 draws of pseudo-event dates.
5. Outliers, MA leave-one-out, window flattening.
6. Sectors, with Benjamini-Hochberg across sector x window.
7. Events with an earnings announcement inside the $[-3,+3]$, $[-4,+4]$ or $[-5,+5]$ window removed.

In [1]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from src.common import (TABLES, INTERIM, EST_LEN, EST_GAP, MIN_OBS, FCOLS, K_VALS, WINDOWS, ar_col,
                        date_col, window_cols, star, p_two_sided, caar_stats, kp_factor, load_sample,
                        load_factors, return_series, ResidualCorrelation)

df = load_sample()
df["CAR_1"] = df[window_cols(-1, 1)].sum(axis=1)
rc = ResidualCorrelation()
series = return_series()
FF = load_factors()
et = df["Event_Type"]
print(f"{len(df)} events")

615 events


## Model comparison

Each event is re-estimated under FF5, FF3 and the market model on the same estimation window.
$\bar\rho$ is estimated once per group and reused across models.

In [2]:
MODELS = {"FF5": FCOLS, "FF3": ["MKT", "SMB", "HML"], "MM": ["MKT"]}
MC_WINDOWS = [(-1, 1), (0, 3), (-3, 3), (-5, 5)]


def event_window(e):
    s = series[e["Region"]].get(e["Identifier"])
    if s is None or not (len(s[s.index <= e["Date"]]) > EST_LEN and len(s[s.index > e["Date"]]) >= 5):
        return None, None
    h = s.to_frame("r").join(FF[e["Region"]], how="inner").dropna()
    d = e["Date"]
    if d not in h.index:
        nxt = h.index[h.index >= d]
        if len(nxt) == 0 or (nxt[0] - d).days > 3:
            return None, None
        d = nxt[0]
    i0 = h.index.get_loc(d)
    if i0 < EST_LEN + 1 or i0 + 5 >= len(h):
        return None, None
    return h, i0


recs = []
for _, e in df.iterrows():
    h, i0 = event_window(e)
    if h is None:
        continue
    est = h.iloc[i0 - EST_LEN: i0 - EST_GAP + 1]
    if len(est) < MIN_OBS:
        continue
    rec = dict(Ticker=e["Ticker"], Date=e["Date"], Region=e["Region"], Event_Type=e["Event_Type"],
               MktCap_USDm=e["MktCap_USDm"])
    y = (est["r"] - est["RF"]).to_numpy(float)
    for m, cols in MODELS.items():
        X = np.column_stack([np.ones(len(est)), est[cols].to_numpy(float)])
        b, *_ = np.linalg.lstsq(X, y, rcond=None)
        rec[f"Sigma_{m}"] = (y - X @ b).std(ddof=len(cols) + 1)
        XtX_inv = np.linalg.pinv(X.T @ X)
        for k1, k2 in MC_WINDOWS:
            xsum = np.concatenate([[k2 - k1 + 1], h.iloc[i0 + k1: i0 + k2 + 1][cols].to_numpy(float).sum(axis=0)])
            rec[f"Lev_{m}_{k1:+d}_{k2:+d}"] = float(xsum @ XtX_inv @ xsum)
        for k in K_VALS:
            z = h.iloc[i0 + k]
            rec[f"{m}_AR_{k}"] = float((z["r"] - z["RF"] - b[0] - np.dot(z[cols].to_numpy(float), b[1:])) * 100)
    recs.append(rec)
mm = pd.DataFrame(recs)
mm.to_csv(INTERIM / "model_event_level.csv", index=False)
print(f"{len(mm)} of {len(df)} events re-estimated")


def caar_model(sub, model, k1, k2, rho):
    ks = [f"{model}_AR_{k}" for k in range(k1, k2 + 1)]
    levc = f"Lev_{model}_{k1:+d}_{k2:+d}"
    z = sub.dropna(subset=ks + [f"Sigma_{model}", levc])
    car = z[ks].sum(axis=1)
    n = len(z)
    sar = car / (z[f"Sigma_{model}"] * 100 * np.sqrt(len(ks) + z[levc]))
    t = sar.mean() / (sar.std(ddof=1) / np.sqrt(n))
    factor, _ = kp_factor(z, rho, n)
    if rho > 0 and np.isfinite(factor):
        t = t * factor
    zv = z.dropna(subset=["MktCap_USDm"])
    w = zv["MktCap_USDm"].to_numpy(float)
    w = w / w.sum()
    vw = float((zv[ks].sum(axis=1).to_numpy() * w).sum())
    return dict(N=n, EW_CAAR=car.mean(), t_EW=t, p_EW=p_two_sided(t, n - 1),
                VW_CAAR=vw)


rows = []
for g, sub in {"Full sample": mm, "US": mm[mm["Region"] == "US"], "EU": mm[mm["Region"] == "EU"],
               "MA": mm[mm["Event_Type"] == "MA"], "CP": mm[mm["Event_Type"] == "CP"]}.items():
    rho = rc.rho_bar(sub)
    for m in MODELS:
        for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-3,+3]", -3, 3), ("[-5,+5]", -5, 5)]:
            r = caar_model(sub, m, k1, k2, rho)
            rows.append({"Group": g, "Model": m, "Window": w, **r, "sig": star(r["p_EW"])})
models = pd.DataFrame(rows)
models.to_csv(TABLES / "model_comparison.csv", index=False)
models[models["Group"] == "US"]

615 of 615 events re-estimated


,Group,Model,Window,N,EW_CAAR,t_EW,p_EW,VW_CAAR,sig
9,US,FF5,"[-1,+1]",545,-0.0666,-0.7620,0.4464,-0.0523,
10,US,FF5,"[-3,+3]",545,-0.4196,-2.7494,0.0062,-0.2618,***
11,US,FF5,"[-5,+5]",545,-0.4383,-2.4945,0.0129,-0.2495,**
12,US,FF3,"[-1,+1]",545,-0.0018,-0.3507,0.7260,0.0150,
13,US,FF3,"[-3,+3]",545,-0.3511,-2.4209,0.0158,-0.3026,**
14,US,FF3,"[-5,+5]",545,-0.4339,-2.3989,0.0168,-0.3930,**
15,US,MM,"[-1,+1]",545,0.0076,0.0694,0.9447,0.0928,
16,US,MM,"[-3,+3]",545,-0.3369,-1.7375,0.0829,-0.0980,*
17,US,MM,"[-5,+5]",545,-0.3093,-1.4798,0.1395,0.0072,


## Pre-2023 / post-2022 split

Third Bridge calls are dated 2018-2023 only, so the split also changes the event mix.

In [3]:
post, us, tb = df["Date"] >= "2023-01-01", df["Region"] == "US", df["third_bridge"]
rows = []
for g, sub in [("Post-2022", df[post]), ("Pre-2023", df[~post]),
               ("Post-2022 ex-TB", df[post & ~tb]), ("Pre-2023 ex-TB", df[~post & ~tb]),
               ("US Post-2022", df[post & us]), ("US Pre-2023", df[~post & us]),
               ("US Post-2022 ex-TB", df[post & us & ~tb]), ("US Pre-2023 ex-TB", df[~post & us & ~tb]),
               ("US Post-2022 TB only", df[post & us & tb]), ("US Pre-2023 TB only", df[~post & us & tb])]:
    rho = rc.rho_bar(sub)
    rows += [{"group": g, "window": w, **caar_stats(sub, k1, k2, rho=rho)} for w, k1, k2 in WINDOWS]
split = pd.DataFrame(rows)
split.to_csv(TABLES / "pre_post_2023.csv", index=False)
split[split["window"] == "[-3,+3]"][["group", "N", "EW_CAAR", "t_used", "p_used"]]

,group,N,EW_CAAR,t_used,p_used
4,Post-2022,397,-0.0653,-0.8623,0.3890
11,Pre-2023,218,-0.6709,-3.2068,0.0015
18,Post-2022 ex-TB,322,-0.2552,-1.4342,0.1525
25,Pre-2023 ex-TB,95,-0.9730,-3.1145,0.0024
32,US Post-2022,359,-0.2888,-1.4701,0.1424
39,US Pre-2023,186,-0.6722,-2.8603,0.0047
46,US Post-2022 ex-TB,292,-0.5200,-2.1296,0.0340
53,US Pre-2023 ex-TB,80,-0.9217,-2.5655,0.0122
60,US Post-2022 TB only,67,0.7191,1.5939,0.1157
67,US Pre-2023 TB only,106,-0.4839,-1.4267,0.1566


## Estimation-window contamination

FF5 is re-estimated over $\tau \in \{-250, \ldots, -20\}$ without the $[-5,+5]$ windows of the firm's other
events. The SCAR omits the forecast-error term. The baseline CAAR on the same events is reported for comparison.

In [4]:
dates_by_firm = df.groupby("firm_key")["Date"].apply(lambda s: np.sort(s.dt.normalize().unique()))
n_prior = 0
for _, e in df.iterrows():
    h = series[e["Region"]][e["Identifier"]].to_frame("r").join(FF[e["Region"]], how="inner").dropna()
    pos = h.index.searchsorted(pd.Timestamp(e[date_col(0)]))
    lo, hi = np.datetime64(h.index[pos - EST_LEN]), np.datetime64(h.index[pos - EST_GAP])
    others = dates_by_firm[e["firm_key"]]
    n_prior += bool(((others >= lo) & (others <= hi)).any())
print(f"Events with an earlier same-firm event inside their estimation window: {n_prior} of {len(df)}")

recs = []
for _, e in df.iterrows():
    s = series[e["Region"]].get(e["Identifier"])
    h = s.to_frame("r").join(FF[e["Region"]], how="inner").dropna() if s is not None else []
    if len(h) < 300:
        continue
    pos = h.index.searchsorted(pd.Timestamp(e[date_col(0)]))
    if pos >= len(h) or pos - EST_LEN < 0 or pos + 5 >= len(h):
        continue
    mask = np.zeros(len(h), dtype=bool)
    mask[pos - EST_LEN: pos - EST_GAP + 1] = True
    for d in dates_by_firm[e["firm_key"]]:
        if pd.Timestamp(d) != e["Date"].normalize():
            p = h.index.searchsorted(pd.Timestamp(d))
            mask[max(0, p - 5): min(len(h), p + 6)] = False
    est = h[mask]
    if len(est) < MIN_OBS:
        continue
    X = np.column_stack([np.ones(len(est)), est[FCOLS].to_numpy(float)])
    y = (est["r"] - est["RF"]).to_numpy(float)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    win = h.iloc[pos - 5: pos + 6]
    ar = ((win["r"] - win["RF"]).to_numpy(float) - np.column_stack([np.ones(11), win[FCOLS].to_numpy(float)]) @ b) * 100
    recs.append(dict(event_id=e["event_id"], Region=e["Region"], Date=e["Date"],
                     Sigma_clean=(y - X @ b).std(ddof=6), **{ar_col(k): ar[k + 5] for k in K_VALS}))
clean = pd.DataFrame(recs)
print(f"{len(clean)} of {len(df)} events re-estimated")

rho_region = {"Full sample": rc.rho_bar(df), "US": rc.rho_bar(df[us]), "EU": rc.rho_bar(df[~us])}
rows = []
for g, sub in [("Full sample", clean), ("US", clean[clean["Region"] == "US"]), ("EU", clean[clean["Region"] == "EU"])]:
    rho = rho_region[g]
    for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-3,+3]", -3, 3), ("[-5,+5]", -5, 5)]:
        ks = window_cols(k1, k2)
        z = sub.dropna(subset=ks)
        car = z[ks].sum(axis=1)
        n = len(z)
        sar = car / (z["Sigma_clean"] * 100 * np.sqrt(len(ks)))
        t = sar.mean() / (sar.std(ddof=1) / np.sqrt(n))
        factor, _ = kp_factor(z, rho, n)
        t = t * factor if (rho > 0 and np.isfinite(factor)) else t
        base = df[df["event_id"].isin(z["event_id"])].dropna(subset=ks)
        rows.append(dict(group=g, window=w, N=n, CAAR_clean=car.mean(), t_used_clean=t,
                         p_used_clean=p_two_sided(t, n - 1),
                         CAAR_baseline_same_events=base[ks].sum(axis=1).mean()))
estwin = pd.DataFrame(rows)
estwin.to_csv(TABLES / "estwin_contamination.csv", index=False)
estwin

Events with an earlier same-firm event inside their estimation window: 339 of 615


592 of 615 events re-estimated


,group,window,N,CAAR_clean,t_used_clean,p_used_clean,CAAR_baseline_same_events
0,Full sample,"[-1,+1]",592,-0.0254,-0.7665,0.4437,-0.0025
1,Full sample,"[-3,+3]",592,-0.3238,-2.5259,0.0118,-0.2715
2,Full sample,"[-5,+5]",592,-0.3426,-2.3956,0.0169,-0.2715
3,US,"[-1,+1]",522,-0.0880,-0.8503,0.3956,-0.0625
4,US,"[-3,+3]",522,-0.4749,-2.8122,0.0051,-0.4162
5,US,"[-5,+5]",522,-0.5207,-2.6661,0.0079,-0.4398
6,EU,"[-1,+1]",70,0.4414,0.1161,0.9079,0.4455
7,EU,"[-3,+3]",70,0.8035,0.3446,0.7315,0.8071
8,EU,"[-5,+5]",70,0.9859,0.4110,0.6823,0.9837


## Placebo

Per firm: FF5 on all days outside $\pm10$ days of its events, residuals for every day.
Each replication draws one pseudo-event day per event, at least 20 trading days from any real
event, and records CAAR[-1,+1]. 500 replications.

In [5]:
resid, sigma, valid_pos = {}, {}, {}
for (base, region, ident), dates in df.groupby(["base", "Region", "Identifier"])["Date"].apply(list).items():
    s = series[region].get(ident)
    h = s.to_frame("r").join(FF[region], how="inner").dropna()
    if len(h) < 300:
        continue
    pos = [h.index.searchsorted(pd.Timestamp(d)) for d in dates]
    excl, near = np.zeros(len(h), dtype=bool), np.zeros(len(h), dtype=bool)
    for p in pos:
        excl[max(0, p - 10): min(len(h), p + 11)] = True
        near[max(0, p - 20): min(len(h), p + 21)] = True
    est = h[~excl]
    if len(est) < MIN_OBS:
        continue
    X = np.column_stack([np.ones(len(est)), est[FCOLS].to_numpy(float)])
    y = (est["r"] - est["RF"]).to_numpy(float)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    valid = np.arange(5, len(h) - 5)
    valid = valid[~near[valid]]
    if len(valid) < 50:
        continue
    X_all = np.column_stack([np.ones(len(h)), h[FCOLS].to_numpy(float)])
    resid[(base, region)] = ((h["r"] - h["RF"]).to_numpy(float) - X_all @ b) * 100
    sigma[(base, region)] = (y - X @ b).std(ddof=6) * 100
    valid_pos[(base, region)] = valid

covered = df[[(b, r) in resid for b, r in zip(df["base"], df["Region"])]]
keys = list(zip(covered["base"], covered["Region"]))
is_ma = (covered["Event_Type"] == "MA").to_numpy()
print(f"{len(resid)} firms, {len(covered)} events covered (MA {is_ma.sum()})")

N_REP = 500
rng = np.random.default_rng(42)
draws = []
for rep in range(N_REP):
    cars, sars = [], []
    for key in keys:
        p = int(rng.choice(valid_pos[key]))
        car = float(resid[key][p - 1: p + 2].sum())
        cars.append(car)
        sars.append(car / (sigma[key] * np.sqrt(3)))
    cars, sars = np.array(cars), np.array(sars)
    sm_ = sars[is_ma]
    draws.append(dict(rep=rep, caar_full=float(cars.mean()),
                      t_full=float(sars.mean() / (sars.std(ddof=1) / np.sqrt(len(sars)))),
                      caar_ma=float(cars[is_ma].mean()),
                      t_ma=float(sm_.mean() / (sm_.std(ddof=1) / np.sqrt(len(sm_))))))
placebo_draws = pd.DataFrame(draws)
placebo_draws.to_csv(TABLES / "placebo_draws.csv", index=False)

rows = []
for g, sub, col in [("Full sample", covered, "caar_full"), ("MA", covered[is_ma], "caar_ma")]:
    actual = caar_stats(sub, -1, 1).get("EW_CAAR", np.nan)
    dist = placebo_draws[col]
    rows.append(dict(group=g, actual_CAAR=actual, placebo_mean=dist.mean(),
                     placebo_sd=dist.std(ddof=1), placebo_p5=dist.quantile(0.05),
                     placebo_p95=dist.quantile(0.95),
                     percentile_of_actual=float((dist < actual).mean())))
placebo = pd.DataFrame(rows)
placebo.to_csv(TABLES / "placebo.csv", index=False)
placebo

174 firms, 615 events covered (MA 25)


,group,actual_CAAR,placebo_mean,placebo_sd,placebo_p5,placebo_p95,percentile_of_actual
0,Full sample,-0.0083,0.0013,0.1163,-0.1990,0.1932,0.46
1,MA,1.1513,0.0070,0.5669,-0.9549,0.9503,0.97


## Outliers, MA leave-one-out, window flattening

Outliers: $|CAR[-1,+1]| > 8\%$. Flattening: CAAR per trading day for nested windows $[-w,+w]$.

In [6]:
outliers = df[df["CAR_1"].abs() > 8][["Company", "Ticker", "Date", "Region", "Event_Type", "CAR_1",
                                      "MktCap_USDm"]].sort_values("CAR_1")
outliers.to_csv(TABLES / "outliers.csv", index=False)

ma = df[et == "MA"]
loo = []
for i in ma.index:
    r = caar_stats(ma.drop(i), -1, 1)
    loo.append({"dropped": f"{ma.loc[i, 'Ticker']} {ma.loc[i, 'Date'].date()}", "CAAR": r.get("EW_CAAR"),
                "p_EW": r.get("p_EW")})
loo = pd.DataFrame(loo).sort_values("CAAR")
loo.to_csv(TABLES / "ma_leave_one_out.csv", index=False)
print(f"MA leave-one-out CAAR range [{loo['CAAR'].min():.3f}, {loo['CAAR'].max():.3f}]")

rows = []
for g, sub in [("Full sample", df), ("US", df[us]), ("EU", df[~us]), ("MA", ma), ("CP", df[et == "CP"])]:
    for w in range(6):
        r = caar_stats(sub, -w, w)
        rows.append({"Group": g, "Window": f"[-{w},+{w}]" if w else "[0,0]", "half_width": w, "N": r.get("N"),
                     "EW_CAAR": r.get("EW_CAAR"), "p_EW": r.get("p_EW"),
                     "per_day": (r.get("EW_CAAR", np.nan) / (2 * w + 1))})
flattening = pd.DataFrame(rows)
flattening.to_csv(TABLES / "flattening.csv", index=False)
flattening.pivot_table(index="half_width", columns="Group", values="per_day")

MA leave-one-out CAAR range [0.580, 1.427]


Group,CP,EU,Full sample,MA,US
half_width,,,,,
0,-0.0913,-0.0223,-0.0486,1.1793,-0.0520
1,0.0058,0.1485,-0.0028,0.3838,-0.0222
2,0.0113,0.1659,-0.0098,0.2083,-0.0324
3,-0.0341,0.1153,-0.0400,0.0452,-0.0599
4,-0.0633,0.0446,-0.0459,0.0623,-0.0575
5,-0.0436,0.0894,-0.0251,0.1505,-0.0398


## Sectors

$\bar\rho$ per sector. Benjamini-Hochberg q-values across all sector x window tests.

In [7]:
rows = []
for sec, sub in df.groupby("Sector"):
    rho = rc.rho_bar(sub)
    for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-5,+5]", -5, 5)]:
        r = caar_stats(sub, k1, k2, rho=rho)
        if r.get("N", 0) >= 3:
            rows.append({"Sector": sec, "Window": w, **r, "sig_EW": star(r.get("p_EW")),
                         "sig_VW": star(r.get("p_VW")), "sig_used": star(r.get("p_used"))})
sector = pd.DataFrame(rows)
sector["q_EW"] = multipletests(sector["p_used"].where(sector["p_used"].notna(), sector["p_EW"]),
                               method="fdr_bh")[1]
sector["sig_EW_fdr"] = sector["q_EW"].apply(star)
sector.to_csv(TABLES / "sector.csv", index=False)
sector[["Sector", "Window", "N", "EW_CAAR", "p_used", "q_EW"]]

,Sector,Window,N,EW_CAAR,p_used,q_EW
0,Communications,"[-1,+1]",53,-0.4066,0.2048,0.4916
1,Communications,"[-5,+5]",53,-0.1904,0.4239,0.5807
2,Consumer,"[-1,+1]",60,0.3658,0.3158,0.5807
3,Consumer,"[-5,+5]",60,0.1970,0.8307,0.8877
4,Financials,"[-1,+1]",46,-0.5245,0.0839,0.4916
5,Financials,"[-5,+5]",46,-1.6836,0.0175,0.2103
6,Healthcare,"[-1,+1]",47,0.3995,0.4128,0.5807
7,Healthcare,"[-5,+5]",47,0.6520,0.4839,0.5807
8,Industrials,"[-1,+1]",35,-0.1866,0.4557,0.5807
9,Industrials,"[-5,+5]",35,-1.0104,0.1525,0.4916


## Earnings announcements inside the wider windows

The earnings screen of notebook 01 removes events within five calendar days of a results date, which is three or four trading days. For each window, the events whose results date falls on one of the window's trading days are removed and the CAAR is tested again, with $\bar\rho$ estimated on the remaining events.

In [8]:
results_date = pd.to_datetime(df["event_id"].map(
    pd.read_csv(INTERIM / "earnings_screen_report.csv").set_index("event_id")["results_date"]))
rows = []
for g, sub in [("Full sample", df), ("US", df[us]), ("EU", df[~us])]:
    for w, k in [("[-3,+3]", 3), ("[-4,+4]", 4), ("[-5,+5]", 5)]:
        days = sub[[date_col(j) for j in range(-k, k + 1)]].apply(pd.to_datetime)
        inside = days.eq(results_date.loc[sub.index], axis=0).any(axis=1)
        kept = sub[~inside]
        rows.append({"group": g, "window": w, "N_removed": int(inside.sum()),
                     **caar_stats(kept, -k, k, rho=rc.rho_bar(kept))})
earnings_window = pd.DataFrame(rows)
earnings_window.to_csv(TABLES / "earnings_window.csv", index=False)
earnings_window[["group", "window", "N_removed", "N", "EW_CAAR", "t_used", "p_used"]]

,group,window,N_removed,N,EW_CAAR,t_used,p_used
0,Full sample,"[-3,+3]",2,613,-0.2909,-2.4979,0.0128
1,Full sample,"[-4,+4]",18,597,-0.4342,-3.0074,0.0027
2,Full sample,"[-5,+5]",27,588,-0.3148,-2.3605,0.0186
3,US,"[-3,+3]",2,543,-0.4324,-2.7986,0.0053
4,US,"[-4,+4]",16,529,-0.5370,-3.0019,0.0028
5,US,"[-5,+5]",23,522,-0.4231,-2.4498,0.0146
6,EU,"[-3,+3]",0,70,0.8071,0.3668,0.7149
7,EU,"[-4,+4]",2,68,0.3659,-0.4982,0.6200
8,EU,"[-5,+5]",4,66,0.5423,-0.1043,0.9172
